# Exercise 05 — Activation Patching (write it yourself)

Same task as `01_notebooks/05_activation_patching.ipynb`, core logic blanked.
Solution: [`../01_notebooks/05_activation_patching.ipynb`](../01_notebooks/05_activation_patching.ipynb).


In [ ]:
import torch
from transformer_lens import HookedTransformer, utils
from functools import partial

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

clean_text = "John and Mary went to the store, John gave a drink to"
corrupted_text = "John and Mary went to the store, Mary gave a drink to"
clean_tokens = model.to_tokens(clean_text)
corrupted_tokens = model.to_tokens(corrupted_text)
mary_token = model.to_single_token(" Mary")
john_token = model.to_single_token(" John")
clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)


## Exercise 1 — the metric

Write `logit_diff(logits)`: at the last token position, return
`logit(" Mary") - logit(" John")` as a plain Python float.


In [ ]:
def logit_diff(logits):
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# self-check
_clean = logit_diff(clean_logits)
_corrupted = logit_diff(corrupted_logits)
assert abs(_clean - 2.2588157653808594) < 1e-2, f"clean logit diff off: got {_clean}"
assert abs(_corrupted - (-3.7372121810913086)) < 1e-2, f"corrupted logit diff off: got {_corrupted}"
print("Exercise 1 passed — clean:", _clean, "corrupted:", _corrupted)


## Exercise 2 — the patch hook

Write `patch_resid_pre(resid_pre, hook, position, clean_cache)`: overwrite
`resid_pre` at the given `position` (all batch elements, all `d_model`
entries) with the corresponding value from `clean_cache`, leaving every
other position untouched. Return the modified tensor (TransformerLens hooks
must return the tensor they receive).

<details>
<summary>Hint</summary>

`hook.name` is the activation name (e.g. `"blocks.3.hook_resid_pre"`) — use
it to index into `clean_cache`. You only need one line:
`resid_pre[:, position, :] = clean_cache[hook.name][:, position, :]`.
</details>


In [ ]:
def patch_resid_pre(resid_pre, hook, position, clean_cache):
    # YOUR CODE HERE
    raise NotImplementedError
    return resid_pre


In [ ]:
# self-check — unit-test the hook function directly, no model run needed
import types

_dummy_hook = types.SimpleNamespace(name="blocks.0.hook_resid_pre")
_start = corrupted_cache["blocks.0.hook_resid_pre"].clone()
_patched = patch_resid_pre(_start, _dummy_hook, position=3, clean_cache=clean_cache)
assert torch.allclose(_patched[:, 3, :], clean_cache["blocks.0.hook_resid_pre"][:, 3, :]), \
    "position 3 should now match the clean run"
assert torch.allclose(_patched[:, 0, :], corrupted_cache["blocks.0.hook_resid_pre"][:, 0, :]), \
    "position 0 should be untouched (still corrupted)"
print("Exercise 2 passed")


## Exercise 3 — the full sweep

For every `(layer, position)` pair, run the corrupted prompt with
`patch_resid_pre` installed at that single point, and record how much of the
clean/corrupted logit-diff gap gets recovered (0 = no recovery, 1 = fully
back to the clean answer). Fill in `results[layer, position]`.

<details>
<summary>Hint</summary>

Use `model.run_with_hooks(corrupted_tokens, fwd_hooks=[(utils.get_act_name("resid_pre", layer), hook_fn)])`
where `hook_fn = partial(patch_resid_pre, position=position, clean_cache=clean_cache)`.
Normalize with `(patched_diff - corrupted_diff) / (clean_diff - corrupted_diff)`.
</details>


In [ ]:
n_layers = model.cfg.n_layers
n_pos = clean_tokens.shape[1]
clean_diff = logit_diff(clean_logits)
corrupted_diff = logit_diff(corrupted_logits)
results = torch.zeros(n_layers, n_pos)

for layer in range(n_layers):
    for position in range(n_pos):
        # YOUR CODE HERE — compute patched_diff and set results[layer, position]
        raise NotImplementedError


In [ ]:
# self-check
assert results.shape == (n_layers, n_pos)
assert results.max().item() > 0.5, f"expected some (layer, position) to recover most of the clean answer, best was {results.max().item():.3f}"
assert results.min().item() < 0.3, "expected some positions to show little to no recovery"
print("Exercise 3 passed — max recovery:", results.max().item())


In [ ]:
import plotly.express as px

str_tokens = model.to_str_tokens(clean_text)
px.imshow(results, x=str_tokens, y=list(range(n_layers)),
          labels=dict(x="position", y="layer", color="recovery"),
          title="Activation patching: resid_pre recovery of clean logit diff",
          color_continuous_scale="RdBu", color_continuous_midpoint=0).show()
